In [ ]:
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchResults
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os

load_dotenv()

api_key = os.environ['UNIFIED_LLM_KEY']
# print(api_key)
base_url = ""

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=256,
    api_key=api_key,
    base_url=base_url
)

In [ ]:
# Alternative approach: Use Wikipedia search instead (no external dependencies)
# Or install using conda which handles SSL differently
import sys
import subprocess

try:
    # Try installing via conda first
    result = subprocess.run(
        ['conda', 'install', '-c', 'conda-forge', 'duckduckgo-search', '-y'],
        capture_output=True,
        text=True,
        timeout=60
    )
    if result.returncode == 0:
        print("✅ Installed duckduckgo-search via conda")
    else:
        print("❌ Conda install failed, trying pip with no SSL verification...")
        # Last resort: pip with no SSL verification (use with caution)
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--trusted-host', 'pypi.org',
             '--trusted-host', 'pypi.python.org', '--trusted-host', 'files.pythonhosted.org',
             'duckduckgo-search'],
            check=True
        )
        print("✅ Installed duckduckgo-search via pip")
except FileNotFoundError:
    print("Conda not found, trying pip...")
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--trusted-host', 'pypi.org',
         '--trusted-host', 'pypi.python.org', '--trusted-host', 'files.pythonhosted.org',
         'duckduckgo-search'],
        check=True
    )
    print("✅ Installed duckduckgo-search via pip")
except Exception as e:
    print(f"❌ Installation failed: {e}")
    print("\n💡 Alternative: Use Wikipedia search instead (see next cell)")

In [ ]:
# Try DuckDuckGo first, fallback to Wikipedia if it fails
try:
    from langchain_community.tools import DuckDuckGoSearchResults
    search_tool = DuckDuckGoSearchResults(max_results=5)
    print("✅ Using DuckDuckGo search")
except ImportError:
    print("⚠️  DuckDuckGo not available, using Wikipedia instead")
    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wikipedia = WikipediaAPIWrapper(top_k_results=3, doc_content_chars_max=500)
    search_tool = WikipediaQueryRun(api_wrapper=wikipedia)


class ResearchState(TypedDict):
    question: str
    search_results: str
    answer: str

def search_web(state: ResearchState):
    """Search the web for information"""
    print(f"🔍 Searching for: {state['question']}")
    results = search_tool.invoke(state["question"])
    return {"search_results": results}

def generate_answer(state: ResearchState):
    print("💭 Generating answer...")
    prompt = f"""
    Question: {state['question']}

    Search Results:
    {state['search_results']}

    Provide a comprehensive answer based on the search results.
    """
    response = llm.invoke(prompt)
    return {"answer": response.content}

workflow = StateGraph(ResearchState)
workflow.add_node("search", search_web)
workflow.add_node("answer", generate_answer)
workflow.add_edge(START, "search")
workflow.add_edge("search", "answer")
workflow.add_edge("answer", END)


checkpoint = InMemorySaver()
graph = workflow.compile(checkpointer=checkpoint)

config = {"configurable": {"thread_id": "research_1"}}

result = graph.invoke({
    "question": "What is LangGraph?",
    "search_results": "",
    "answer": ""
}, config)

print("\nAnswer:")
print(result["answer"])